In [0]:
%run ../../config/config

In [0]:
%run ../../config/sqlconfig

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DoubleType, TimestampType, DecimalType, DateType, BooleanType
from pyspark.sql import functions as F

In [0]:
class GoldSalesEnriched:
    def __init__(self, spark):
        self.spark = spark

        self.catalog = CONFIG['catalog']['name']
        self.schema =  CONFIG['catalog']['schema']

        # up-stream table name
        self.silver_customer_tbl = TABLES['silver']['customer']
        self.silver_product_tbl = TABLES['silver']['product']
        self.silver_sales_tbl = TABLES['silver']['sales']

        # down-stream table name
        self.downstream_table_name = TABLES['gold']['sales_enriched']

        #path
        self.checkpoint_path = CONFIG['path']['checkpoint']

    def read_upstream_table(self, table_name, is_stream=True):
        if is_stream:
            return self.spark.readStream.table(f"{self.catalog}.{self.schema}.{table_name}")
        else:
            return self.spark.read.table(f"{self.catalog}.{self.schema}.{table_name}")
    
    def upsert_to_gold(self, microBatchDF, batchId):
        try:
            print(f"Batch {batchId} count: {microBatchDF.count()}")
            microBatchDF.show(5, False)

            microBatchDF.createOrReplaceTempView('v_sales_enriched')
            microBatchDF.sparkSession.sql(f"""
                MERGE INTO {self.catalog}.{self.schema}.{self.downstream_table_name} as t
                USING v_sales_enriched as s    
                ON t.sales_id = s.sales_id
                WHEN NOT MATCHED THEN INSERT *                  
            """)
        except Exception as e:
            print(f"Error: {e}")
        

    def process_stream(self):
        # fact = streaming
        sales_tbl = self.read_upstream_table(self.silver_sales_tbl, is_stream=True)

        # dimensions = batch (IMPORTANT)
        product_tbl = self.read_upstream_table(self.silver_product_tbl, is_stream=False)
        customer_tbl = self.read_upstream_table(self.silver_customer_tbl, is_stream=False)
        return (
            sales_tbl.alias('sales')
            .join(product_tbl.alias('product'), F.col('sales.product_key') == F.col('product.product_key'), 'left')
            .join(customer_tbl.alias('customer'), F.col('sales.customer_key') == F.col('customer.customer_key'), 'left')
            .select(
                F.col("sales.sales_id"),
                F.to_timestamp("sales.transaction_ts", "dd-MM-yyyy HH:mm").alias("transaction_ts"),
                F.to_date(F.col("sales.date_key").cast("string"), "yyyyMMdd").alias("date_key"),
                F.col("sales.store_key"),
                F.col("sales.payment_key"),
                F.col("sales.quantity"),
                F.col("sales.unit_price"),
                F.col("sales.discount_pct"),
                F.col("sales.net_sales_amount"),
                F.col("sales.tax_amount"),
                F.col("product.Product_name").alias("product_name"),  
                F.col("product.category"),
                F.col("product.brand"),
                F.col("customer.customer_name"),
                F.col("customer.gender"),
                F.col("customer.city"),
            )
        )
    

    # --------------------------------
    # Only for droping table
    # ---------------------------------
    #def drop_table(self):
    #    dbutils.fs.rm(f"{self.checkpoint_path}/_{self.downstream_table_name}", True)

    def write_stream(self, df):
        try:            
            query = (
                df.writeStream.option("checkpointLocation", f"{self.checkpoint_path}/_{self.downstream_table_name}")
                .foreachBatch(self.upsert_to_gold)
                .trigger(availableNow=True)
                .start() 
            )
            query.awaitTermination()
        except Exception as e:
            print(f"Error: {e}")
    def run(self):
        df = self.process_stream()
        self.write_stream(df)


In [0]:
obj = GoldSalesEnriched(spark)
obj.run()

In [0]:
#obj.drop_table()

In [0]:
%sql
select *from gold_sales_enriched

In [0]:
%sql
-- select * from silver_product

In [0]:
%sql
-- select * from silver_fact_sales

In [0]:
%sql
-- select * from silver_product

In [0]:
%sql
-- select *from v_sales_enriched